按顺序运行，默认只显示最终答案。将首个代码单元格的 `VERBOSE` 改为 `True` 可查看学习日志。
模型配置读取根目录 `.env`；提交前清空输出，实际结果记录到 `docs/progress/`。


In [ ]:
from my_agent.output import debug, quiet_stream, run_quietly, set_verbose, with_style, RESPONSE_STYLE

VERBOSE = False  # 排错或学习内部过程时改成 True
set_verbose(VERBOSE)

from my_agent.config import PROJECT_ROOT, load_project_env

env_path = load_project_env()
debug("配置文件：", env_path)


In [ ]:
from typing import List, Dict, Any

In [ ]:
import os
from pathlib import Path
from my_agent.config import load_project_env

# 统一使用项目根目录的 .env
env_path = PROJECT_ROOT / ".env"

debug("当前工作目录：", Path.cwd())
debug(".env 文件存在：", env_path.is_file())

if not env_path.is_file():
    raise FileNotFoundError(f"找不到配置文件：{env_path}")

# 加载根目录配置，保留内核已有的环境变量
_ = load_project_env()

# 只显示是否读到，不打印密钥
for key in ("LLM_MODEL_ID", "LLM_API_KEY", "LLM_BASE_URL"):
    debug(key, "已读取" if os.getenv(key) else "未读取")

In [ ]:
class Memory:
    def __init__(self):
        self.records : List[Dict[str,Any]] = []

    def add_record(self,record_type:str,content:str):
        record = {"type":record_type,"content":content}
        self.records.append(record)
        debug(f"📝 记忆已更新，新增一条 '{record_type}' 记录。")
        

    def get_trajectory(self) ->str:
        part=[]
        for record in self.records:         
            if record["type"] == "execution":
                part.append(f"--- 上一轮尝试 (代码) ---\n{record['content']}")
            if record["type"] == "reflection":
                part.append(f"--- 评审员反馈 ---\n{record['content']}")
        return "\n".join(part)

    def get_last_execution(self):
        for record in reversed(self.records):
            if record["type"]=="execution":
                return record['content']
        return None


In [ ]:
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。

要求: {task}

请直接输出代码，不要包含任何额外的解释。
"""

REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出其在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}
```

请分析该代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案来显著提升性能。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议（例如，使用筛法替代试除法）。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

如果无需改进，整段回复必须严格为：无需改进
不要添加引号、标点、Markdown 格式或其他解释。

如果需要改进，直接输出具体问题与改进建议。

请直接输出你的反馈，不要包含任何额外的解释。
"""

REFINE_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。你正在根据一位代码评审专家的反馈来优化你的代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈：
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""

In [ ]:
from my_agent.llm_client import HelloAgentsLLM
class RefectionAgent:
    def __init__(self,llm_client,max_iterations=3):
        self.llm_client=llm_client
        self.memory=Memory()
        self.max_iteration= max_iterations

    def run(self,task: str):
        debug(f"\n--- 开始处理任务 ---\n任务: {task}")
        #1、初次调用
        debug("\n--- 正在进行初始尝试 ---")
        initial_prompt=INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code=self._get_llm_response(initial_prompt)
        self.memory.add_record("execution",initial_code)

        #反思和优化的循环
        for i in range(self.max_iteration):
            debug("\n-> 正在进行反思...")
            last_code=self.memory.get_last_execution()
            reflection_prompt = REFLECT_PROMPT_TEMPLATE.format(task=task,code=last_code)
            feedback = self._get_llm_response(reflection_prompt)
            self.memory.add_record("reflection",feedback)

            if feedback.strip() == "无需改进":
                debug("\n✅ 反思认为代码已无需改进，任务完成。")
                break

            debug("\n-> 正在进行优化...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(task=task,last_code_attempt=last_code,feedback=feedback)
            new_code=self._get_llm_response(refine_prompt)
            self.memory.add_record("execution",new_code)

        final_code=self.memory.get_last_execution()
        debug(f"\n--- 任务完成 ---\n最终生成的代码:\n```python\n{final_code}\n```")
        return final_code

    def _get_llm_response(self,prompt) -> str:
        messages = [{"role":"user","content":prompt}]
        return self.llm_client.think(messages=messages)

    
        

In [ ]:
llm_client = HelloAgentsLLM()
Agent = RefectionAgent(llm_client=llm_client)
code = Agent.run("编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。")
print(code)
